# MISP 2.5 REST API workshop

Read-only cells for an isolated training instance. First follow [README](README.md) and install `examples/requirements.txt` in this kernel's Python environment (Python 3.11+). Set `MISP_URL` and `MISP_API_KEY` outside the notebook. Use trusted TLS; `REQUESTS_CA_BUNDLE` can name your CA bundle. Never store keys or production responses in the notebook.

The fixtures use reserved `.example` names. Empty results are valid if no fixture has been created.

In [ ]:
from pathlib import Path
import sys

course = Path.cwd()
if not (course / "examples" / "search_attributes.py").is_file():
    course = course / "a.7-rest-API"
sys.path.insert(0, str(course / "examples"))
from search_attributes import MISPClient, iter_attributes

client = MISPClient.from_env()
print("MISP version:", client.version()["version"])


## One raw JSON page

Attribute search returns `response.Attribute`, while event search returns `response` as an array of `Event` wrappers. This helper validates the attribute envelope before returning rows. The server may cap the requested limit.

In [ ]:
import json

query = json.loads((course / "examples" / "attribute-search.json").read_text())
page = client.attributes(query)
print("First page rows:", len(page))
for attribute in page[:3]:
    print(attribute["uuid"], attribute["type"])


## Bounded pagination

Continue through short pages until an empty page. A role cap can make every page shorter than the requested limit. A maximum-page failure or repeated UUID means the export is incomplete. Stable ordering does not freeze a changing dataset. This sample excludes post-query filters such as warninglist enforcement.

In [ ]:
rows = list(iter_attributes(client, query, page_size=100, max_pages=20))
print("Completed bounded traversal:", len(rows))
# No checkpoint is advanced by this notebook.


## The same task through PyMISP

Use `PyMISP`, `type_attribute`, `include_event_tags` and `pythonify=True`. The result is a list of typed objects; an error can instead be a dictionary. This example deliberately fetches only one page.

In [ ]:
from pymisp_search import search_page

typed_page = search_page()
print("PyMISP first page:", len(typed_page))
for attribute in typed_page[:3]:
    print(attribute.uuid, attribute.type)


## Compare and discuss

- Compare raw and typed UUIDs for the same page.
- Repeat using the other organisation's key: which private fixture disappears?
- Why does `to_ids=0` differ from omitting the filter?
- Why is the event's publication time different from the attribute's observation time?
- Explain how a consumer would handle edits, soft deletions and lost visibility.

For deliberate create/edit/delete operations, follow the UUID-based lab in the README. These notebook cells never perform those writes. Clear all outputs before sharing.